In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             mean_squared_error, r2_score)
import matplotlib.pyplot as plt
import seaborn as sns

# ── Load merged data ──────────────────────────────────────────────────
df = pd.read_csv('oregon_fires_weather_merged.csv', parse_dates=['report_date'])

# ── Features ──────────────────────────────────────────────────────────
features = ['TEMP','MAX','MIN','TEMP_RANGE','DEWP','VPD_PROXY',
            'WDSP','MXSPD','PRCP','PRCP_30DAY','MONTH','DOY','FIRE_SEASON']

df_model = df[features + ['total_acres']].dropna()
print(f"Modeling rows: {len(df_model):,}")

X = df_model[features]
y_size = df_model['total_acres']

# ── Fire size categories for classification ───────────────────────────
# Class 0: small (< 10 acres), Class 1: medium (10–100), Class 2: large (100+)
def size_class(acres):
    if acres < 10:   return 0
with acres < 100:  return 1
    else:            return 2

df_model['SIZE_CLASS'] = df_model['total_acres'].apply(size_class)
print("\nFire size class distribution:")
print(df_model['SIZE_CLASS'].value_counts().sort_index()
      .rename({0:'Small (<10 ac)', 1:'Medium (10-100 ac)', 2:'Large (100+ ac)'}))

y_class = df_model['SIZE_CLASS']

# ── Train/test split ──────────────────────────────────────────────────
X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
    X, y_class, y_size, test_size=0.2, random_state=42
)

# ════════════════════════════════════════════════════════════════════
# MODEL 1: Classification — predict fire size class
# ════════════════════════════════════════════════════════════════════
print("\n── Training Classification Model ──")
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, yc_train)
yc_pred = clf.predict(X_test)

print(classification_report(yc_test, yc_pred,
      target_names=['Small','Medium','Large']))

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(yc_test, yc_pred)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[0],
            xticklabels=['Small','Medium','Large'],
            yticklabels=['Small','Medium','Large'],
            cmap='Blues')
axes[0].set_title('Classification: Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# Feature importance - classifier
feat_imp_c = pd.Series(clf.feature_importances_, index=features).sort_values()
feat_imp_c.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Classification: Feature Importance')
axes[1].set_xlabel('Importance')
plt.tight_layout()
plt.savefig('classification_results.png', dpi=150)
plt.show()
print("Saved classification_results.png")

# ════════════════════════════════════════════════════════════════════
# MODEL 2: Regression — predict fire size in acres
# ════════════════════════════════════════════════════════════════════
print("\n── Training Regression Model ──")

# Log transform acres (highly skewed)
yr_train_log = np.log1p(yr_train)
yr_test_log  = np.log1p(yr_test)

reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
reg.fit(X_train, yr_train_log)
yr_pred_log = reg.predict(X_test)
yr_pred     = np.expm1(yr_pred_log)

rmse = np.sqrt(mean_squared_error(yr_test, yr_pred))
r2   = r2_score(yr_test_log, yr_pred_log)
print(f"RMSE: {rmse:.2f} acres")
print(f"R²  : {r2:.3f} (on log scale)")

# Regression plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs predicted
axes[0].scatter(np.log1p(yr_test), yr_pred_log, alpha=0.3, s=10, color='steelblue')
axes[0].plot([0, 10], [0, 10], 'r--')
axes[0].set_xlabel('Actual log(acres)')
axes[0].set_ylabel('Predicted log(acres)')
axes[0].set_title(f'Regression: Actual vs Predicted (R²={r2:.3f})')

# Feature importance - regressor
feat_imp_r = pd.Series(reg.feature_importances_, index=features).sort_values()
feat_imp_r.plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Regression: Feature Importance')
axes[1].set_xlabel('Importance')
plt.tight_layout()
plt.savefig('regression_results.png', dpi=150)
plt.show()
print("Saved regression_results.png")

IndentationError: unexpected indent (3688416400.py, line 28)